# PII Masking with NLP

In [ ]:
!pip --quiet install langdetect 

In [ ]:
import csv
from langdetect import detect
import pickle

In [ ]:
from utils import *

### View Data

#### Note: we had a badline our in the dataset
Bad lines: 1
Bad lines occur at row numbers: [42759]

In [ ]:
bad_row_numbers = []
with open('PII43k.csv', 'r') as f:
	reader = csv.reader(f)
	header = next(reader)
	expected_len = len(header)
	for idx, row in enumerate(reader, start=2):  # start=2 accounts for the header line
		if len(row) != expected_len:
			bad_row_numbers.append(idx)

print("Bad lines occur at row numbers:", bad_row_numbers)

In [ ]:
df_full = pd.read_csv('PII43k.csv', on_bad_lines='skip')

In [ ]:
print(df_full["Template"][0])
print(df_full["Filled Template"][0])

In [ ]:


cleaned_matches, unique_matches = get_template_tokens(df_full)

print(unique_matches)
print(len(unique_matches))

print(cleaned_matches)
print(len(cleaned_matches))


In [ ]:
import re

def count_sentences(text):
	if not isinstance(text, str):
		return 0
	sentences = re.split(r'(?<=[.!?])\s+', text.strip())
	# Filter out any empty strings
	return len([s for s in sentences if s])

df_full['sentence_count'] = df_full['Filled Template'].apply(count_sentences)
df_full[['Filled Template', 'sentence_count']].head()

In [ ]:

cleaned_name_tokens, name_tokens = get_token_tokens(df_full)
print(name_tokens)
print(len(name_tokens))
print(cleaned_name_tokens)
print(len(cleaned_name_tokens))

In [ ]:
# Count occurrences of each cleaned token in df_full
counts = {}
for token in cleaned_matches:
    pattern = r'\[' + token + r'_\d+\]'
    counts[token] = int(df_full["Template"].dropna().str.count(pattern).sum())

# Convert to a DataFrame and sort by count for a nicer display
counts_df = pd.DataFrame(list(counts.items()), columns=['Token', 'Count']).sort_values(by='Count', ascending=False)
counts_df

# Count occurrences of each cleaned token in the "Tokens" column of df_full
token_counts = {}
for token in cleaned_name_tokens:
    # Each row in "Tokens" is a list, so count token appearances per row.
    token_counts[token] = int(df_full["Tokens"].dropna().apply(lambda lst: lst.count(token)).sum())

# Convert counts to a DataFrame and sort for display
token_counts_df = pd.DataFrame(list(token_counts.items()), columns=['Token', 'Count']).sort_values(by='Count', ascending=False)
token_counts_df

counts_df_renamed = counts_df.rename(columns={'Token':'template_entity', 'Count':'template_count'})
token_counts_df_renamed = token_counts_df.rename(columns={'Token':'token_entity', 'Count':'token_count'})

merged_df = pd.concat([counts_df_renamed.reset_index(drop=True),
                       token_counts_df_renamed.reset_index(drop=True)], axis=1)
merged_df

In [ ]:

df_full.head()

In [ ]:
sextype_examples = df_full[df_full['Template'].str.contains(r'\[FULLNAME_1\]', na=False)]

if sextype_examples.empty:
	print("No rows found with the token [SEXTYPE]")
else:
	num_rows = min(10, len(sextype_examples))
	for i in range(num_rows):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)


In [ ]:
cleaned_matches, unique_matches = get_template_tokens(df_full)

def replace_unique_tokens(text, tokens):
	for token in tokens:
		# Match either an underscore with one or more digits or with 'N'
		pattern = r'\[' + token + r'_(?:\d+|N)\]'
		# Replace with the token in square brackets (e.g., "[NAME]")
		text = re.sub(pattern, f'[{token}]', text)
	return text

df_full['Template'] = df_full['Template'].apply(lambda t: replace_unique_tokens(t, cleaned_matches))

# view the first 5 rows of the 'Template' column
df_full['Template'].head()

In [ ]:
sextype_examples = df_full[df_full['Template'].str.contains(r'\[SEXTYPE\]', na=False)]

if sextype_examples.empty:
	print("No rows found with the token [SEXTYPE]")
else:
	num_rows = min(10, len(sextype_examples))
	for i in range(num_rows):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)


In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[SEX\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
sextype_examples = df_full[df_full['Template'].str.contains(r'\[GENDER\]', na=False)]

if sextype_examples.empty:
	print("No rows found with the token [SEXTYPE]")
else:
	num_rows = min(20, len(sextype_examples))
	for i in range(num_rows):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)


In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[BUILDINGNUMBER\]')]

# Display the first few examples with both the Template and Filled Template columns.
# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[JOBAREA\]')]

# Display the first few examples with both the Template and Filled Template columns.
# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[DISPLAYNAME\]')]

# Display the first few examples with both the Template and Filled Template columns.
# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[ACCOUNTNAME\]')]

# Display the first few examples with both the Template and Filled Template columns.
# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[IP\]')]

# Display the first few examples with both the Template and Filled Template columns.
# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
sextype_examples = df_full[df_full['Template'].str.contains(r'\[IPV6\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
sextype_examples = df_full[df_full['Template'].str.contains(r'\[IPV4\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[MAC\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
sextype_examples = df_full[df_full['Template'].str.contains(r'\[JOBDESCRIPTOR\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
sextype_examples = df_full[df_full['Template'].str.contains(r'\[JOBTYPE\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[JOBTITLE\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[ORDINALDIRECTION\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 7):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[NUMBER\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[MASKEDNUMBER\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0,20):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[AMOUNT\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[ACCOUNTNUMBER\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[CURRENCYSYMBOL\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[CURRENCYNAME\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[CURRENCY\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[CURRENCYCODE\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[BITCOINADDRESS\]')]

# Display the first few examples with both the Template and Filled Template columns.
if sextype_examples.empty:
	print("No rows found with the token ")
else:
	for i in range (0, 10):
			print("")
			print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
			print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
			print("")
			print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[ETHEREUMADDRESS\]')]

# Display the first few examples with both the Template and Filled Template columns.

for i in range (0, 10):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[LITECOINADDRESS\]')]

# Display the first few examples with both the Template and Filled Template columns.

for i in range (0, 10):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[CREDITCARDCVV\]')]

# Display the first few examples with both the Template and Filled Template columns.

for i in range (0, 10):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[PIN\]')]

# Display the first few examples with both the Template and Filled Template columns.

for i in range (0, 10):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)

In [ ]:
# Filter df_full for rows where the '[SEXTYPE]' token appears in the Template column.
sextype_examples = df_full[df_full['Template'].str.contains(r'\[CREDITCARDISSUER\]')]

# Display the first few examples with both the Template and Filled Template columns.

for i in range (0, 10):
		print("")
		print(sextype_examples[['Template', 'Filled Template']]["Template"].iloc[i])
		print(sextype_examples[['Template', 'Filled Template']]["Filled Template"].iloc[i])
		print("")
		print("="*120)

In [ ]:
# 
# def safe_detect(text):
# 	try:
# 		return detect(text)
# 	except Exception:
# 		return None
# 
# # Detect language for each text in the 'Filled Template' column
# df_full['language'] = df_full['Filled Template'].apply(safe_detect)
# 
# # Display a sample of the results
# print(df_full[['Filled Template', 'language']].head())
# 
# # Save the updated DataFrame to a pickle file.
# with open("df_full_language.pkl", "wb") as f:
# 	pickle.dump(df_full, f)
# 

In [ ]:


# Load the DataFrame from the pickle file.
with open("df_full_language.pkl", "rb") as f:
	df_full = pickle.load(f)

# Display a sample of the loaded results.
print(df_full[['Filled Template', 'language']].head())

In [ ]:
unique_languages = df_full['language'].unique()
print("Languages present in 'language' column:", unique_languages)

In [ ]:
lang_map = {
	'en': 'English',
	'fr': 'French',
	'ca': 'Catalan',
	'nl': 'Dutch',
	'es': 'Spanish'
}

for lang in unique_languages:
	language_name = lang_map.get(lang, lang)
	examples = df_full[df_full['language'] == lang]["Filled Template"].head(5)
	print(f"Examples for {language_name}:")
	for text in examples:
		print("-", text)
	print("\n" + "-"*40 + "\n")

In [ ]:
import re

emoji_pattern = re.compile(
	"["
	"\U0001F600-\U0001F64F"  # emoticons
	"\U0001F300-\U0001F5FF"  # symbols & pictographs
	"\U0001F680-\U0001F6FF"  # transport & map symbols
	"\U0001F1E0-\U0001F1FF"  # flags
	"]+", flags=re.UNICODE)

# Check for emojis in the 'Filled Template' column of df_full
df_with_emoji = df_full[df_full['Filled Template'].apply(lambda text: bool(emoji_pattern.search(text)))]
if df_with_emoji.empty:
	print("No emojis found in the dataset.")
else:
	print("Emojis found in the dataset. Examples:")
	print(df_with_emoji['Filled Template'].head())